# Maxwell–Boltzmann Distribution: Simulation and Equilibrium Analysis

This project tests whether randomly paired elastic collisions transform a nonequilibrium gas into the three-dimensional Maxwell speed distribution. It includes the full Python engine, conservation tests, reproducible experiments, plots, and an optional interactive parameter explorer.

**Run from top to bottom in Google Colab or Jupyter.** All code is contained here. The small CSV alongside this notebook is a reference benchmark, not required input. Saved graphical outputs are omitted to keep the download small; running the notebook regenerates them.

## 1. Setup
Python 3.10+; NumPy 2.x and Matplotlib. Run the following installation cell if needed. The optional explorer also uses ipywidgets. No credentials or external datasets are needed.

In [ ]:
# Run this cell only if dependencies are missing:
# %pip install "numpy>=2,<3" "matplotlib>=3.7,<4" ipywidgets
import numpy as np
import matplotlib.pyplot as plt
import csv, json
from pathlib import Path
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

: 

## 2. Mathematical model
For molecular mass $m$, temperature $T$, and Boltzmann constant $k_B$, the equilibrium speed density is

$$f(v)=4\pi\left(\frac{m}{2\pi k_BT}\right)^{3/2}v^2e^{-mv^2/(2k_BT)},\quad v\ge0.$$

The most probable, mean, and RMS speeds are $\sqrt{2k_BT/m}$, $\sqrt{8k_BT/(\pi m)}$, and $\sqrt{3k_BT/m}$. They scale as $\sqrt{T/m}$.

For each randomly selected pair, preserve center velocity $\mathbf c=(\mathbf v_i+\mathbf v_j)/2$ and relative speed $g=|\mathbf v_i-\mathbf v_j|$. Draw a direction $\hat{\mathbf n}$ uniformly on the sphere and set

$$\mathbf v_i'=\mathbf c+(g/2)\hat{\mathbf n},\qquad\mathbf v_j'=\mathbf c-(g/2)\hat{\mathbf n}.$$

This preserves pair momentum and kinetic energy. One sweep pairs each particle at most once. Collision partners are independent of position: this is a homogeneous stochastic model, not geometric hard-sphere dynamics. Sweeps are not physical seconds.

## 3. Simulation engine
Six gases and four initial conditions: equal-speed shell, opposing beams, hot/cold populations of one species, and equilibrium-like Gaussian velocities. Velocities are centered and energy-normalized at initialization. For odd N, centering slightly perturbs the equal-speed shell.

In [ ]:
"""Three-dimensional, homogeneous equal-mass elastic collision model (SI units)."""
from __future__ import annotations
import math
import numpy as np

KB = 1.380649e-23  # J/K, exact SI
NA = 6.02214076e23  # mol^-1, exact SI
GASES = {"He": 4.002602, "Ne": 20.1797, "N2": 28.0134, "O2": 31.9988, "Ar": 39.948, "Xe": 131.293}  # g/mol
MODES = ("shell", "beams", "mixture", "equilibrium")


def scale(temperature: float, molar_mass: float) -> float:
    """One-component thermal standard deviation in m/s; mass input in g/mol."""
    if not all(math.isfinite(x) and x > 0 for x in (temperature, molar_mass)):
        raise ValueError("Temperature and molar mass must be finite and positive")
    return math.sqrt(KB * temperature / (molar_mass / (1000 * NA)))


def speed_pdf(speed, temperature=300., molar_mass=GASES["N2"]):
    s = scale(temperature, molar_mass)
    v = np.asarray(speed, dtype=float)
    q = np.maximum(v, 0) / s
    return np.sqrt(2 / np.pi) * q**2 * np.exp(-q**2 / 2) / s


def speed_cdf(speed, temperature=300., molar_mass=GASES["N2"]):
    q = np.maximum(np.asarray(speed, dtype=float), 0) / scale(temperature, molar_mass)
    erf = np.vectorize(math.erf, otypes=[float])(q / np.sqrt(2))
    return np.clip(erf - np.sqrt(2 / np.pi) * q * np.exp(-q**2 / 2), 0, 1)


def characteristic_speeds(temperature=300., molar_mass=GASES["N2"]):
    s = scale(temperature, molar_mass)
    return {"most_probable": math.sqrt(2) * s, "mean": math.sqrt(8 / math.pi) * s, "rms": math.sqrt(3) * s}


def ks_distance(speeds, temperature, molar_mass):
    """Descriptive ECDF distance; NOT an iid KS p-value for correlated collisions."""
    v = np.sort(np.asarray(speeds))
    if not len(v):
        raise ValueError("At least one speed required")
    f = speed_cdf(v, temperature, molar_mass)
    n = len(v)
    return float(max(np.max(np.arange(1, n + 1) / n - f), np.max(f - np.arange(n) / n)))


class GasSimulation:
    """Uniform random pairing with isotropic scattering, not hard-sphere MD.

    Each sweep randomly pairs all particles once. For odd N one particle rests.
    Velocities are stored as dimensionless v / sqrt(k_B T / m).
    """
    def __init__(self, n=4000, temperature=300., molar_mass=GASES["N2"], seed=42, mode="shell"):
        self.sigma = scale(temperature, molar_mass)
        if isinstance(n, bool) or not isinstance(n, (int, np.integer)) or n < 4:
            raise ValueError("n must be an integer >= 4")
        if mode not in MODES:
            raise ValueError(f"mode must be one of {MODES}")
        if isinstance(seed, bool) or not isinstance(seed, (int, np.integer)) or not 0 <= seed <= 2**32 - 1:
            raise ValueError("seed must be an integer in [0, 2^32-1]")
        self.n, self.temperature, self.molar_mass = n, temperature, molar_mass
        self.seed, self.mode, self.sweeps, self.collisions = int(seed), mode, 0, 0
        self.rng = np.random.default_rng(seed)
        if mode == "shell":
            # Antipodal pairs make exact equal-speed, zero-momentum states for even N.
            a = self.rng.normal(size=((n + 1)//2, 3))
            a /= np.linalg.norm(a, axis=1)[:, None]
            q = np.concatenate((a, -a))[:n]
        elif mode == "beams":
            q = np.zeros((n, 3)); q[:, 0] = np.where(np.arange(n) % 2, -1, 1)
        else:
            q = self.rng.normal(size=(n, 3))
            if mode == "mixture":
                q[:n//2] *= math.sqrt(.2)
                q[n//2:] *= math.sqrt(1.8)
        q -= q.mean(axis=0)
        q *= math.sqrt(3 / np.mean(np.sum(q*q, axis=1)))
        self.q = q
        self.initial_energy = float(np.sum(q*q))
        self.initial_momentum = q.sum(axis=0).copy()

    @property
    def velocities(self):
        return self.q * self.sigma

    @property
    def speeds(self):
        return np.linalg.norm(self.q, axis=1) * self.sigma

    def step(self, sweeps=1):
        if isinstance(sweeps, bool) or not isinstance(sweeps, (int, np.integer)) or sweeps < 0:
            raise ValueError("sweeps must be a nonnegative integer")
        for _ in range(sweeps):
            pairs = self.rng.permutation(self.n)[:2*(self.n//2)].reshape(-1, 2)
            a, b = pairs[:, 0], pairs[:, 1]
            center = (self.q[a] + self.q[b]) / 2
            half_relative = np.linalg.norm(self.q[a] - self.q[b], axis=1) / 2
            # A normalized Gaussian vector is uniform over the sphere.
            direction = self.rng.normal(size=(len(a), 3))
            direction /= np.linalg.norm(direction, axis=1)[:, None]
            delta = half_relative[:, None] * direction
            self.q[a], self.q[b] = center + delta, center - delta
            self.sweeps += 1
            self.collisions += len(a)
        return self

    def diagnostics(self):
        s = self.speeds
        energy = float(np.sum(self.q*self.q))
        mass = self.molar_mass / (1000 * NA)
        return {"sweep": self.sweeps, "collisions": self.collisions,
                "temperature_K": self.temperature * energy / (3*self.n),
                "mean_speed_m_s": float(s.mean()), "rms_speed_m_s": float(np.sqrt(np.mean(s*s))),
                "energy_J": .5 * mass * self.sigma**2 * energy,
                "relative_energy_drift": (energy - self.initial_energy) / self.initial_energy,
                "normalized_momentum_drift": float(np.linalg.norm(self.q.sum(axis=0) - self.initial_momentum) / (self.n * math.sqrt(3))),
                "ks_distance": ks_distance(s, self.temperature, self.molar_mass)}


## 4. Automated validation
Tests cover normalization, moments, conservation, relaxation, reproducibility, scaling, odd particle counts, and input validation. A small CDF distance is a descriptive fit measure, not an iid KS p-value.

In [ ]:
import unittest
import numpy as np

class PhysicsTests(unittest.TestCase):
    def test_pdf_normalization_and_moments(self):
        for T,M in [(50,GASES['He']),(300,GASES['N2']),(1500,GASES['Xe'])]:
            v=np.linspace(0,10*scale(T,M),100001);p=speed_pdf(v,T,M);t=characteristic_speeds(T,M)
            self.assertAlmostEqual(np.trapezoid(p,v),1,places=9)
            self.assertAlmostEqual(np.trapezoid(v*p,v)/t['mean'],1,places=9)
            self.assertAlmostEqual(np.sqrt(np.trapezoid(v*v*p,v))/t['rms'],1,places=9)
            self.assertEqual(float(speed_pdf(-1,T,M)),0)
            self.assertEqual(float(speed_cdf(-1,T,M)),0)
            self.assertAlmostEqual(float(speed_cdf(v[-1],T,M)),1,places=10)
    def test_conservation_and_relaxation(self):
        for mode in ('shell','beams','mixture','equilibrium'):
            s=GasSimulation(n=12000,seed=42,mode=mode);d0=s.diagnostics();s.step(60);d=s.diagnostics()
            self.assertLess(abs(d['relative_energy_drift']),1e-12)
            self.assertLess(d['normalized_momentum_drift'],1e-13)
            self.assertLess(d['ks_distance'],.025)
            if mode!='equilibrium':self.assertLess(d['ks_distance'],d0['ks_distance'])
    def test_reproducibility(self):
        a=GasSimulation(seed=71).step(12);b=GasSimulation(seed=71).step(12)
        np.testing.assert_array_equal(a.q,b.q)
    def test_odd_population(self):
        s=GasSimulation(n=501).step(9)
        self.assertEqual(s.collisions,250*9)
        self.assertLess(abs(s.diagnostics()['relative_energy_drift']),1e-12)
    def test_scaling(self):
        a=GasSimulation(temperature=300,molar_mass=20);b=GasSimulation(temperature=600,molar_mass=20)
        np.testing.assert_allclose(b.speeds,a.speeds*np.sqrt(2))
        c=GasSimulation(temperature=300,molar_mass=80)
        np.testing.assert_allclose(c.speeds,a.speeds/2)
    def test_initial_conditions(self):
        s=GasSimulation(mode='shell')
        self.assertLess(np.std(s.speeds),1e-10)
        b=GasSimulation(mode='beams')
        self.assertTrue(np.all(b.q[:,1:]==0))
    def test_invalid_inputs(self):
        for kw in ({'temperature':0},{'molar_mass':-1},{'n':3},{'n':4.5},{'mode':'oops'},{'seed':-1},{'temperature':float('nan')}):
            with self.assertRaises(ValueError):GasSimulation(**kw)
        with self.assertRaises(ValueError):GasSimulation().step(-1)

result = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(PhysicsTests))
assert result.wasSuccessful(), "Physics validation failed"

## 5. Reproducible experiment
Start with nitrogen at 300 K and 12,000 particles. Edit the parameters below to explore another run. The plotting function is reused by the interactive explorer.

In [ ]:
def run_experiment(n=12000, temperature=300., gas="N2", mode="shell", seed=42, sweeps=60):
    sim = GasSimulation(n=n, temperature=temperature, molar_mass=GASES[gas], seed=seed, mode=mode)
    initial = sim.speeds.copy()
    history = [sim.diagnostics()]
    for _ in range(sweeps):
        sim.step()
        history.append(sim.diagnostics())
    return sim, initial, history


def plot_experiment(sim, initial, history):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    upper = max(initial.max(), sim.speeds.max(), 5*sim.sigma)
    bins = np.linspace(0, upper, 55)
    axes[0].hist(initial, bins=bins, density=True, histtype="step", color="gray", label="Initial")
    axes[0].hist(sim.speeds, bins=bins, density=True, alpha=.6, color="teal", label="Final")
    v = np.linspace(0, upper, 500)
    axes[0].plot(v, speed_pdf(v, sim.temperature, sim.molar_mass), color="darkorange", label="Theory")
    axes[0].set(title="Speed distribution", xlabel="Speed (m/s)", ylabel="Density (s/m)")
    axes[0].legend()
    axes[1].plot([h["sweep"] for h in history], [h["ks_distance"] for h in history], color="teal")
    axes[1].set(title="Approach to equilibrium", xlabel="Collision sweep", ylabel="CDF distance D", ylim=(0,None))
    axes[2].plot([h["sweep"] for h in history], [h["relative_energy_drift"] for h in history])
    axes[2].set(title="Energy conservation", xlabel="Collision sweep", ylabel="Relative energy drift")
    plt.show()
    return fig

sim, initial, history = run_experiment()
example_figure = plot_experiment(sim, initial, history)
print(json.dumps(history[-1], indent=2))

### Interpretation
Elastic collisions redistribute the speeds while preserving the total energy and momentum to floating-point precision. The final histogram fluctuates around the theoretical density. A finite sample will not exactly reproduce a smooth distribution, and its CDF distance need not decrease on every sweep. The distribution of velocities is also constrained by the conserved quantities.

## 6. Independent seeds
Run 10 seeds for each of three populations. The shaded bands are 10th–90th empirical percentiles across runs, not confidence intervals. This cell generates the included benchmark snapshot.

In [ ]:
benchmark_rows = []
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
for n in (500, 2000, 8000):
    curves = []
    for seed in range(10):
        trial, _, records = run_experiment(n=n, seed=seed, sweeps=40)
        curves.append([r["ks_distance"] for r in records])
        benchmark_rows.append({"particles": n, "seed": seed, **records[-1]})
    curves = np.asarray(curves)
    ax.plot(np.arange(41), curves.mean(axis=0), label=f"N={n:,}")
    ax.fill_between(np.arange(41), np.quantile(curves,.1,axis=0), np.quantile(curves,.9,axis=0), alpha=.15)
    print(f"N={n:5d}: mean final D={curves[:,-1].mean():.6f}")
ax.set(xlabel="Collision sweep", ylabel="CDF distance D", yscale="log", title="Relaxation across 10 seeds")
ax.legend()
plt.show()
benchmark_figure = fig

### Interpretation
Larger samples generally track the theoretical distribution more closely. The mean final distance across seeds is more informative than a single trajectory. Three sample sizes do not establish an asymptotic convergence rate. Adjacent sweeps are correlated and should not be treated as independent replications.

## 7. Temperature and molecular mass
These curves are analytical predictions, not additional simulated trajectories. The tests above independently verify the corresponding velocity scaling in the simulation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
v = np.linspace(0, 2400, 700)
for T in (150, 300, 600):
    axes[0].plot(v, speed_pdf(v,T,GASES["N2"]), label=f"{T} K")
for gas in ("He", "N2", "Xe"):
    axes[1].plot(v, speed_pdf(v,300,GASES[gas]), label=gas)
for ax, title in zip(axes, ["Nitrogen at different temperatures", "Different gases at 300 K"]):
    ax.set(title=title, xlabel="Speed (m/s)", ylabel="Density (s/m)")
    ax.legend()
plt.show()
scaling_figure = fig

## 8. References and project format

- [Maxwell velocity distribution — University of Texas at Austin](https://farside.ph.utexas.edu/teaching/sm1/Thermalhtml/node87.html)
- [Boltzmann constant — NIST](https://physics.nist.gov/cgi-bin/cuu/Value?k)